#ALINEAMIENTO MULTIPLE CLUSTAL OMEGA

1. Extraer Accession IDs desde el archivo .txt de BLAST

In [2]:
import re
from Bio import Entrez
import time
import os

# ---------- CONFIGURACIÓN ----------
archivo_blast = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\Blast\Blastn_02_07_25\Blastn_virus_bacteria.txt"  # Cambia esto por tu archivo
top_hits = 5
Entrez.email = "fgarciao2206@gmail.com"  # Reemplaza por tu correo real

# Ruta personalizada para guardar las secuencias FASTA
directorio_salida = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\top_sequences_blastn"
os.makedirs(directorio_salida, exist_ok=True)

# ---------- EXTRAER ACCESSION IDs ----------
def extraer_accessions(archivo_blast, top_hits=5):
    with open(archivo_blast, "r", encoding="utf-8") as file:
        contenido = file.read()

    queries = contenido.split("Query #")
    accesiones_por_query = []

    for query in queries[1:]:  # Omitimos el encabezado
        matches = re.findall(r'\b([A-Z]{1,2}_?\d+\.\d+)\b', query)
        accesiones = list(dict.fromkeys(matches))[:top_hits]  # Únicos y top N
        accesiones_por_query.append(accesiones)

    return accesiones_por_query

# ---------- DESCARGAR SECUENCIAS ----------
def descargar_secuencias(accession_list, nombre_archivo):
    with open(nombre_archivo, "w") as output_handle:
        for acc in accession_list:
            try:
                handle = Entrez.efetch(db="nucleotide", id=acc, rettype="fasta", retmode="text")
                seq_record = handle.read()
                output_handle.write(seq_record)
                time.sleep(0.5)  # Espera para no saturar NCBI
            except Exception as e:
                print(f"Error al descargar {acc}: {e}")

# ---------- EJECUCIÓN ----------
ids_por_query = extraer_accessions(archivo_blast, top_hits)

for i, accesiones in enumerate(ids_por_query, start=1):
    archivo_salida = os.path.join(directorio_salida, f"query_{i}_top5.fasta")
    print(f"Descargando secuencias para Query #{i}...")
    descargar_secuencias(accesiones, archivo_salida)

print("\n✅ ¡Todos los archivos FASTA han sido generados en la carpeta especificada!")


Descargando secuencias para Query #1...
Descargando secuencias para Query #2...
Descargando secuencias para Query #3...
Descargando secuencias para Query #4...
Descargando secuencias para Query #5...
Descargando secuencias para Query #6...
Descargando secuencias para Query #7...
Descargando secuencias para Query #8...
Descargando secuencias para Query #9...
Descargando secuencias para Query #10...
Descargando secuencias para Query #11...
Descargando secuencias para Query #12...
Descargando secuencias para Query #13...

✅ ¡Todos los archivos FASTA han sido generados en la carpeta especificada!


2. Extracción de segmentos en formato FASTA - Genomas completos

In [5]:
import os
from Bio import Entrez, SeqIO

# Configuración
Entrez.email = "tu_email@institucion.edu"  # ¡Obligatorio! Usa tu email real
directorio_salida = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\top_sequences_blastn"
os.makedirs(directorio_salida, exist_ok=True)

# Diccionario con las regiones a extraer (mismo que antes)
regiones_por_query = {
    "Query2_Ebola": [
        ("OR084925.1", 287, 337), ("OR084925.1", 303, 337), 
        ("OR084918.1", 287, 337), ("OR084918.1", 303, 337),
        ("OR084860.1", 287, 337), ("OR084860.1", 303, 337),
        ("OR084932.1", 287, 337), ("OR084932.1", 303, 337),
        ("OR084890.1", 287, 337), ("OR084890.1", 303, 337)

    ],
    "Query6_Ebola": [
        ("KY785940.1", 4368, 4717), ("KY785940.1", 4219, 4372), ("KY785940.1", 12619, 12688),
        ("KY471125.1", 4368, 4717), ("KY471125.1", 4219, 4372), ("KY471125.1", 12619, 12688),
        ("KY785948.1", 4368, 4717), ("KY785948.1", 4219, 4372), ("KY785948.1", 12619, 12688),
        ("KY785965.1", 4368, 4717), ("KY785965.1", 4219, 4372), ("KY785965.1", 12619, 12688),
        ("KY785969.1", 4368, 4717), ("KY785969.1", 4219, 4372), ("KY785969.1", 12619, 12688)
    ],
    "Query9_Ebola": [
        ("KY785940.1", 4425, 4641), ("KY785940.1", 4217, 4424), ("KY785940.1", 4642, 4717),
        ("KY785939.1", 4425, 4641), ("KY785939.1", 4217, 4424), ("KY785939.1", 4642, 4717),
        ("KY785965.1", 4425, 4641), ("KY785965.1", 4217, 4424), ("KY785965.1", 4642, 4717),
        ("KY471125.1", 4425, 4641), ("KY471125.1", 4217, 4424), ("KY471125.1", 4642, 4717),
        ("KY785948.1", 4425, 4641), ("KY785948.1", 4217, 4424), ("KY785948.1", 4642, 4717) 
    ],
    "Query10_Salmonella": [
        ("CP140732.2", 4094133, 4094710), ("CP140732.2", 4094133, 4094710),
        ("CP183506.1", 887637, 888214), ("CP183506.1", 887634, 888214),
        ("CP176561.1", 4075490, 4076067),
        ("CP183583.1", 3916322, 3916899),
        ("CP176707.1", 758652, 759229)
    ],
    "Query11_Salmonella": [
        ("CP176685.1", 757665, 757961), ("CP176685.1", 757459, 757611), ("CP176685.1", 757958, 758095), ("CP176685.1", 757611, 757665),
        ("CP087533.1", 3660324, 3660620), ("CP087533.1", 3660674, 3660826), ("CP087533.1", 3660190, 3660327), ("CP087533.1", 3660620, 3660674),
        ("CP130151.1", 4352710, 4353006), ("CP130151.1", 4353060, 4353212), ("CP130151.1", 4352576, 4352713), ("CP130151.1", 4353006, 4353060),
        ("CP176561.1", 4076838, 4077134), ("CP176561.1", 4077188, 4077340), ("CP176561.1", 4076704, 4076841), ("CP176561.1", 4077134, 4077188),
        ("CP087538.1", 2588198, 2588494), ("CP087538.1", 2588548, 2588700), ("CP087538.1", 2588064, 2588201), ("CP087538.1", 2588494, 2588548)
    ]
}


def extraer_y_formatear_regiones(regiones, nombre_archivo):
    """Extrae regiones desde NCBI y guarda con formato personalizado"""
    with open(nombre_archivo, "w") as handle:
        for acc, start, end in regiones:
            try:
                # Descargar la secuencia completa para obtener la descripción
                handle_seq = Entrez.efetch(
                    db="nucleotide",
                    id=acc,
                    rettype="gb",
                    retmode="text"
                )
                record = SeqIO.read(handle_seq, "gb")
                descripcion = record.description
                
                # Descargar solo la región de interés
                handle_region = Entrez.efetch(
                    db="nucleotide",
                    id=acc,
                    rettype="fasta",
                    strand=1,
                    seq_start=start,
                    seq_stop=end
                )
                secuencia = handle_region.read().split("\n", 1)[1].replace("\n", "")
                
                # Escribir en el formato deseado
                handle.write(f">{acc}:{start}-{end} {descripcion}\n{secuencia}\n")
                print(f"✅ Descargado y formateado: {acc} ({start}-{end})")
                
            except Exception as e:
                print(f"❌ Error con {acc}: {e}")

# Procesamiento principal
print("🔍 Iniciando descarga y formateo de secuencias...")
for query, regiones in regiones_por_query.items():
    # Extraer el número de query (ej: "Query2_Ebola" -> "2")
    query_num = query.split('_')[0].replace('Query', '')
    # Crear el nuevo nombre de archivo en minúsculas
    archivo_fasta = os.path.join(directorio_salida, f"query_{query_num}_top5.fasta")
    print(f"\n📥 Procesando {query} ({len(regiones)} regiones)...")
    extraer_y_formatear_regiones(regiones, archivo_fasta)

print(f"\n🎯 ¡Proceso completado! Archivos en:\n{directorio_salida}")

🔍 Iniciando descarga y formateo de secuencias...

📥 Procesando Query2_Ebola (10 regiones)...
✅ Descargado y formateado: OR084925.1 (287-337)
✅ Descargado y formateado: OR084925.1 (303-337)
✅ Descargado y formateado: OR084918.1 (287-337)
✅ Descargado y formateado: OR084918.1 (303-337)
✅ Descargado y formateado: OR084860.1 (287-337)
✅ Descargado y formateado: OR084860.1 (303-337)
✅ Descargado y formateado: OR084932.1 (287-337)
✅ Descargado y formateado: OR084932.1 (303-337)
✅ Descargado y formateado: OR084890.1 (287-337)
✅ Descargado y formateado: OR084890.1 (303-337)

📥 Procesando Query6_Ebola (15 regiones)...
✅ Descargado y formateado: KY785940.1 (4368-4717)
✅ Descargado y formateado: KY785940.1 (4219-4372)
✅ Descargado y formateado: KY785940.1 (12619-12688)
✅ Descargado y formateado: KY471125.1 (4368-4717)
✅ Descargado y formateado: KY471125.1 (4219-4372)
✅ Descargado y formateado: KY471125.1 (12619-12688)
✅ Descargado y formateado: KY785948.1 (4368-4717)
✅ Descargado y formateado: KY

3. Multinalineamiento con Clustal Omega - Cleaned Sequences + top 5 blastn alignment

In [8]:
import os
import subprocess
from Bio import SeqIO

# Rutas
ruta_querys = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\identifiacion_patogeno_individual_sequences"
ruta_hits = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\top_sequences_blastn"
ruta_output = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Alineamientos"
os.makedirs(ruta_output, exist_ok=True)

def buscar_sequence_original(nombre_query, carpeta_querys):
    """
    Dado un nombre como 'query_1_top5', retorna el path completo a 'sequence_1.fasta'
    """
    if "query_" in nombre_query and "_top5" in nombre_query:
        numero = nombre_query.replace("query_", "").replace("_top5", "")
        archivo_esperado = f"sequence_{numero}.fasta"
        path = os.path.join(carpeta_querys, archivo_esperado)
        if os.path.exists(path):
            return path
        else:
            return None
    return None

# Procesar cada archivo de hits
for archivo_hits in os.listdir(ruta_hits):
    if archivo_hits.endswith(".fasta"):
        nombre_base = os.path.splitext(archivo_hits)[0]  # Ej: query_1_top5
        path_hits = os.path.join(ruta_hits, archivo_hits)

        # Buscar secuencia original
        path_query = buscar_sequence_original(nombre_base, ruta_querys)
        
        print(f"\n🧪 Procesando: {nombre_base}")
        print(f"🔍 Buscando archivo: {path_query}")
        if path_query is not None:
            print(f"📁 Existe: {os.path.exists(path_query)}")
        else:
            print("📁 No se encontró ruta válida (None)")

        if not path_query:
            print(f"❌ Archivo original no encontrado para {nombre_base}. Saltando...")
            continue

        # Crear archivo combinado
        archivo_combinado = os.path.join(ruta_output, f"{nombre_base}_combined.fasta")
        with open(archivo_combinado, "w") as out_fasta:
            # Escribir la secuencia original (query)
            for record in SeqIO.parse(path_query, "fasta"):
                SeqIO.write(record, out_fasta, "fasta")
            # Escribir los top hits
            for record in SeqIO.parse(path_hits, "fasta"):
                SeqIO.write(record, out_fasta, "fasta")

        # Archivo alineado
        archivo_alineado = os.path.join(ruta_output, f"{nombre_base}_alignment.aln")
        comando = [
            "clustalo",
            "-i", archivo_combinado,
            "-o", archivo_alineado,
            "--outfmt=clustal",
            "--force",
            "--wrap=80"
        ]

        try:
            subprocess.run(comando, check=True)
            print(f"✅ Alineamiento completado: {archivo_alineado}")
        except subprocess.CalledProcessError as e:
            print(f"❌ Error al alinear {nombre_base}: {e}")

        # Opcional: borrar el archivo combinado si no lo necesitas
        # os.remove(archivo_combinado)




🧪 Procesando: query_10_top5
🔍 Buscando archivo: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\identifiacion_patogeno_individual_sequences\sequence_10.fasta
📁 Existe: True
✅ Alineamiento completado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Alineamientos\query_10_top5_alignment.aln

🧪 Procesando: query_11_top5
🔍 Buscando archivo: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\identifiacion_patogeno_individual_sequences\sequence_11.fasta
📁 Existe: True
✅ Alineamiento completado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Alineamientos\query_11_t